In [2]:
# RB 3·4를 포함한 환자의 원본 전체 시퀀스
# 행 = 원본 patient_id, 열 = seq1..seqn, 값 = RB 등급
import os
import pandas as pd

CANDIDATES = [
    "/shared/home/mai/JeongGeon/Private/CXR/unified_labels.csv",  # 서버
    r"C:\Users\USER\Downloads\unified_labels.csv",               # 로컬
]
SRC = next(p for p in CANDIDATES if os.path.exists(p))

pd.set_option(
    "display.max_rows", 200,
    "display.max_columns", 25,
    "display.width", 220,
)

df = pd.read_csv(SRC, encoding="utf-8-sig")

df["year"] = (
    df["patient_id"]
    .astype(str)
    .str[:2]
    .map({"24": 2024, "26": 2026})
)

# 각 환자가 RB 3과 RB 4를 포함하는지 확인
patient_rb = df.groupby("patient_id")["RB"].agg(
    has_rb3=lambda x: x.eq(3).any(),
    has_rb4=lambda x: x.eq(4).any(),
)

rb3_patients = int(patient_rb["has_rb3"].sum())
rb4_patients = int(patient_rb["has_rb4"].sum())
rb34_patients = int(
    (patient_rb["has_rb3"] & patient_rb["has_rb4"]).sum()
)

# RB 3 또는 4를 한 장이라도 가진 환자
hit = set(
    patient_rb.index[
        patient_rb["has_rb3"] | patient_rb["has_rb4"]
    ]
)

# 해당 환자의 전체 시퀀스
sub = df[df["patient_id"].isin(hit)].copy()


def pivot(s):
    """
    행 = 원본 patient_id
    열 = seq1..seqn
    값 = RB 정수
    시퀀스 길이 밖 = 빈칸
    """
    t = s.pivot(
        index="patient_id",
        columns="seq",
        values="RB",
    )

    max_seq = int(t.columns.max())
    t = t.reindex(columns=range(1, max_seq + 1))
    t.columns = [f"seq{c}" for c in t.columns]

    # 실제 RB 값은 정수로 유지하고 결측값은 빈 문자열로 표시
    t = t.astype("Int64")
    t = t.astype(object).where(t.notna(), "")

    return t.sort_index()


seq_rb34 = pivot(sub)

print(f"원본: {SRC}")

print(
    f"전체 {len(df)}장 · 환자 {df['patient_id'].nunique()}명 "
    f"(2024 {(df['year'] == 2024).sum()}장/"
    f"{df.loc[df['year'] == 2024, 'patient_id'].nunique()}명, "
    f"2026 {(df['year'] == 2026).sum()}장/"
    f"{df.loc[df['year'] == 2026, 'patient_id'].nunique()}명)"
)

print(
    f"\nRB 3·4 포함 환자 {len(hit)}명 · "
    f"그 환자들의 전체 영상 {len(sub)}장 "
    f"(그중 RB 3·4는 {int(sub['RB'].isin([3, 4]).sum())}장)"
)

print(
    f"  RB 3 포함 환자: {rb3_patients}명\n"
    f"  RB 4 포함 환자: {rb4_patients}명\n"
    f"  RB 3과 4 모두 포함 환자: {rb34_patients}명"
)

for yr in (2024, 2026):
    s = sub[sub["year"] == yr]

    yr_patient_rb = s.groupby("patient_id")["RB"].agg(
        has_rb3=lambda x: x.eq(3).any(),
        has_rb4=lambda x: x.eq(4).any(),
    )

    yr_rb3_patients = int(yr_patient_rb["has_rb3"].sum())
    yr_rb4_patients = int(yr_patient_rb["has_rb4"].sum())
    yr_rb34_patients = int(
        (yr_patient_rb["has_rb3"] & yr_patient_rb["has_rb4"]).sum()
    )

    print(
        f"  {yr}: 환자 {s['patient_id'].nunique()}명 · 영상 {len(s)}장 "
        f"(RB3 {int((s['RB'] == 3).sum())}장/"
        f"{yr_rb3_patients}명, "
        f"RB4 {int((s['RB'] == 4).sum())}장/"
        f"{yr_rb4_patients}명, "
        f"둘 다 포함 {yr_rb34_patients}명)"
    )

sequence_lengths = sub.groupby("patient_id").size()

print(
    f"\n시퀀스 길이 {sequence_lengths.min()}~"
    f"{sequence_lengths.max()}프레임 "
    f"· 표 {seq_rb34.shape[0]}행 × {seq_rb34.shape[1]}열 "
    f"· 값 {int((seq_rb34 != '').sum().sum())}칸"
)

display(seq_rb34)

원본: /shared/home/mai/JeongGeon/Private/CXR/unified_labels.csv
전체 1305장 · 환자 114명 (2024 718장/68명, 2026 587장/46명)

RB 3·4 포함 환자 78명 · 그 환자들의 전체 영상 898장 (그중 RB 3·4는 380장)
  RB 3 포함 환자: 75명
  RB 4 포함 환자: 35명
  RB 3과 4 모두 포함 환자: 32명
  2024: 환자 45명 · 영상 471장 (RB3 135장/42명, RB4 68장/19명, 둘 다 포함 16명)
  2026: 환자 33명 · 영상 427장 (RB3 122장/33명, RB4 55장/16명, 둘 다 포함 16명)

시퀀스 길이 7~20프레임 · 표 78행 × 20열 · 값 898칸


,seq1,seq2,seq3,seq4,seq5,seq6,seq7,seq8,seq9,seq10,seq11,seq12,seq13,seq14,seq15,seq16,seq17,seq18,seq19,seq20
patient_id,,,,,,,,,,,,,,,,,,,,
24_0407A,2,1,2,3,1,3,2,3,2,2,,,,,,,,,,
24_0407C,1,1,2,2,2,1,2,3,1,1,2,,,,,,,,,
24_0407D,2,1,3,2,1,2,3,2,3,,,,,,,,,,,
24_0407F,2,3,3,3,3,3,2,3,3,3,3,4,,,,,,,,
24_0407G,3,3,0,1,1,0,4,1,1,0,2,1,,,,,,,,
24_0407H,2,1,2,2,1,1,3,1,2,1,1,1,1,,,,,,,
24_0407J,4,3,4,4,4,4,4,3,3,2,,,,,,,,,,
24_0506B,3,3,4,4,4,4,4,4,3,2,4,,,,,,,,,
24_0506D,2,2,1,3,1,1,1,1,1,,,,,,,,,,,
